---
title: 'Informe General — Assessment Radmedis'
date: '2026-03-10'
author: 'Farid Sayago'
lang: 'es'
---

# Informe General: Analisis de Cartera de Clientes

**Fecha:** 2026-03-10  
**Autor:** Farid Sayago  
**Alcance:** 10 clientes — Base de Datos Assessment Radmedis  
**Periodo analizado:** Febrero — Octubre 2024

**Tecnologias utilizadas:**
- Python 3.12
- pandas — manipulacion y analisis de datos tabulares
- matplotlib — visualizacion de datos
- openpyxl — lectura/escritura de archivos Excel
- Jupyter Lab — entorno interactivo de analisis

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from IPython.display import display

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_excel(DATA_DIR / 'Assessment_AnalistaDatos_Ejercicio.xlsx', sheet_name='Base de Datos')

def clasificar_cliente(row):
    if pd.isna(row['Fecha de Pago']) and row['Monto Pagado'] == 0:
        return 'En mora'
    elif pd.isna(row['Fecha de Pago']) and row['Monto Pagado'] > 0:
        return 'Pago sin fecha'
    else:
        return 'Pagado con fecha'

df['Estado'] = df.apply(clasificar_cliente, axis=1)

## Resumen Ejecutivo

El analisis cubre una cartera de 10 clientes con registros de pago para el periodo febrero–octubre 2024. Se identificaron dos clientes en situacion de mora (sin fecha de pago y monto igual a cero), tres clientes con pagos registrados pero sin fecha de cobro confirmada, y cinco clientes con pagos formalizados.

El total recaudado asciende a **1,100,000 COP**. Utilizando un proxy de deuda de 250,000 COP por cliente (total supuesto: 2,500,000 COP), la tasa de recuperacion estimada es del **44%**. Esta cifra es aproximada dado que no se cuenta con la deuda original real por cliente.

Los meses de mayor actividad de pago fueron agosto 2024 (250,000 COP) y febrero 2024 (100,000 COP). No se detectaron errores tipograficos en nombres ni duplicados en identificadores de clientes.

## KPIs de Cartera

In [ ]:
total_recaudado = df['Monto Pagado'].sum()
deuda_proxy = 2_500_000
pct_recuperacion = total_recaudado / deuda_proxy * 100

clientes_mora = df[df['Estado'] == 'En mora']
clientes_sin_fecha = df[df['Estado'] == 'Pago sin fecha']
clientes_pagado = df[df['Estado'] == 'Pagado con fecha']

kpis = pd.DataFrame({
    'KPI': [
        'Total clientes',
        'Total recaudado (COP)',
        'Deuda proxy total (COP)',
        'Tasa de recuperacion (%)',
        'Clientes en mora',
        'Clientes con pago sin fecha',
        'Clientes pagados con fecha'
    ],
    'Valor': [
        len(df),
        f'{total_recaudado:,.0f}',
        f'{deuda_proxy:,.0f}',
        f'{pct_recuperacion:.1f}% *',
        len(clientes_mora),
        len(clientes_sin_fecha),
        len(clientes_pagado)
    ]
})

display(kpis.style.hide(axis='index').set_properties(**{'text-align': 'left'}))
print('\n* Aproximacion basada en proxy de 250,000 COP por cliente. Requiere deuda original real para calculo preciso.')

## Tabla Resumen de Clientes

In [ ]:
tabla_resumen = df[['ID Cliente', 'Nombre', 'Fecha de Pago', 'Monto Pagado', 'Estado']].copy()
tabla_resumen['Fecha de Pago'] = tabla_resumen['Fecha de Pago'].dt.strftime('%Y-%m-%d').fillna('Sin registro')
tabla_resumen['Monto Pagado'] = tabla_resumen['Monto Pagado'].apply(lambda x: f'{x:,.0f}')

display(tabla_resumen.style.hide(axis='index').set_properties(**{'text-align': 'left'}))

## Visualizaciones

In [ ]:
colores = {'En mora': '#d62728', 'Pago sin fecha': '#ff7f0e', 'Pagado con fecha': '#2ca02c'}
df_sorted = df.sort_values('Monto Pagado', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    df_sorted['Nombre'],
    df_sorted['Monto Pagado'],
    color=[colores[e] for e in df_sorted['Estado']],
    edgecolor='white',
    linewidth=0.8
)
ax.set_title('Monto Pagado por Cliente', fontsize=12, fontweight='bold')
ax.set_xlabel('Cliente')
ax.set_ylabel('Monto Pagado (COP)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30, ha='right')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in colores.items()]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'informe_montos.png', bbox_inches='tight')
plt.show()

## Desglose Mensual de Pagos

In [ ]:
df_con_fecha = df.dropna(subset=['Fecha de Pago']).copy()
df_con_fecha['Mes'] = df_con_fecha['Fecha de Pago'].dt.to_period('M').astype(str)

desglose_mes = df_con_fecha.groupby('Mes').agg(
    Clientes=('ID Cliente', 'count'),
    Monto_Total=('Monto Pagado', 'sum')
).reset_index()
desglose_mes.columns = ['Mes', 'N° Clientes', 'Monto Total (COP)']
desglose_mes['Monto Total (COP)'] = desglose_mes['Monto Total (COP)'].apply(lambda x: f'{x:,.0f}')

print('Nota: Solo se incluyen meses con pagos registrados. Meses sin actividad se omiten.')
display(desglose_mes.style.hide(axis='index'))

# Visualizacion
desglose_plot = df_con_fecha.groupby('Mes')['Monto Pagado'].sum().reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(desglose_plot['Mes'], desglose_plot['Monto Pagado'], color='#5b8db8', edgecolor='white')
ax.set_title('Recaudacion Mensual (2024)', fontsize=12, fontweight='bold')
ax.set_xlabel('Mes')
ax.set_ylabel('Monto Total (COP)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'informe_desglose_mensual.png', bbox_inches='tight')
plt.show()

## Conclusiones y Recomendaciones

**Conclusiones:**

1. El 20% de los clientes (C001, C006) se encuentran en mora, sin registros de pago ni monto cobrado. Representan un riesgo de incobrabilidad directa.

2. El 30% de los clientes (C004, C007, C009) presenta pagos registrados pero sin fecha de cobro confirmada, lo que dificulta la trazabilidad temporal y el analisis de flujo de caja real.

3. La tasa de recuperacion estimada del 44% es baja bajo el proxy utilizado. Sin la deuda original real, no es posible determinar si este porcentaje es critico o aceptable en el contexto del negocio.

4. Los pagos registrados se concentran en cinco meses discretos del año, sin patron de regularidad mensual, lo que sugiere ausencia de esquemas de pago estructurados o cuotas periodicas.

**Recomendaciones:**

1. Obtener el monto de deuda original por cliente para calcular la tasa de recuperacion real y priorizar gestiones de cobro.

2. Implementar registro obligatorio de fecha en todos los pagos para habilitar analisis de flujo de caja confiables.

3. Definir un proceso formal de seguimiento para clientes en mora (C001, C006) con plazos y responsables asignados.

4. Establecer un esquema de pagos periodicos para los clientes actuivos a fin de suavizar la recaudacion a lo largo del año.